# Forzy + K-Means: do sinal bruto ao classificador preditivo

Aplicacao do **pipeline da aula (Projeto Ford)** a telemetria real de dois motores
monitorados por sensor IO-Link.

## O pipeline da aula
1. carregar a base historica;
2. escolher features para a analise;
3. visualizacao inicial;
4. preparar os dados;
5. metodo do cotovelo para apoiar a escolha de `k`;
6. treinar o K-Means;
7. interpretar a segmentacao;
8. **transformar os clusters em target**;
9. treinar e avaliar um classificador supervisionado.

## A regra critica (slide 7 da aula)
> *"Nao misturar passado conhecido com futuro desconhecido."*

Na Ford sao duas tabelas: **Base 1** (historico completo, comportamento observado) e
**Base 2** (apenas o que se sabe no momento da compra).

Aqui nao temos duas tabelas - temos um **stream temporal**. A traducao e direta:

|            | Ford                           | Forzy                                     |
|------------|--------------------------------|-------------------------------------------|
| **Base 1** | comportamento pos-venda (24 m) | **janela futura**: proximos 10 min        |
| **Base 2** | dados no momento da compra     | **janela passada**: ultimos 30 s a 10 min |

O K-Means enxerga o **futuro**. O classificador so enxerga o **presente e o passado**.
Os dois conjuntos de variaveis sao **disjuntos** - e isso que impede o pipeline de ser
circular (treinar um modelo para reproduzir o proprio K-Means).

**Pergunta rapida:** nesta etapa existe target `y`?
**Resposta:** nao. A Etapa 1 e **aprendizado nao supervisionado**. O `y` e *construido*
por ela e so existe a partir da Etapa 2.

## 1) Importar bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
RANDOM_STATE = 42

## 2) Carregar a telemetria bruta

Arquivo exportado do gateway IO-Link: dois ports (motor 1 e motor 2), amostragem
irregular, periodo de ~4 h.

> No Colab, troque a celula por `files.upload()`.

In [ ]:
ARQUIVO = "History_32026-05-19T11-46-10-920.csv"

df_raw = pd.read_csv(
    ARQUIVO, sep=";", header=2,
    names=["ts", "pdi_motor1", "pdi_motor2",
           "v1", "a1", "t1", "v2", "a2", "t2"],
)
df_raw["ts"] = pd.to_datetime(df_raw["ts"], errors="coerce")

print("Linhas e colunas:", df_raw.shape)
print("Periodo:", df_raw.ts.min(), "->", df_raw.ts.max())
print("Duracao total:", round((df_raw.ts.max() - df_raw.ts.min()).total_seconds() / 60, 1), "min")
display(df_raw.head())

## 3) EDA - as colunas do Excel sao redundantes

**O que esta etapa faz:** checa, antes de qualquer modelo, se as variaveis carregam
informacao independente. E o item *"a escala das variaveis e comparavel?"* do
checklist de EDA da aula.

> **Pergunta rapida:** posso jogar as 6 colunas no K-Means?
> **Resposta:** so depois de verificar. K-Means usa **distancia** - variaveis
> duplicadas contam **duas vezes** e distorcem o agrupamento.

In [ ]:
cols_excel = ["v1", "a1", "t1", "v2", "a2", "t2"]
for c in cols_excel:
    df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce")

corr = df_raw[cols_excel].corr()

plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={"label": "correlacao"})
plt.title("Correlacao entre as colunas exportadas para o Excel")
plt.tight_layout()
plt.show()

pca_check = PCA().fit(StandardScaler().fit_transform(df_raw[cols_excel].dropna()))
var = pca_check.explained_variance_ratio_
print("Variancia explicada por componente:", var.round(3))
print(f"Apenas 2 componentes explicam {var[:2].sum()*100:.1f}% da variancia.")

### Leitura

`velocidade` e `aceleracao` correlacionam **1.00**. Os motores 1 e 2 tambem.
Seis colunas, mas **~2 dimensoes reais de informacao**: *quanto vibra* e *quao quente esta*.

Isso nao e ruido a ser tratado - e uma pista de que o arquivo Excel **perdeu informacao**
na exportacao. Vamos ao payload bruto.

## 4) Decodificar o payload IO-Link

**O que esta etapa faz:** le o campo `PDI Data Byte Array`, que o Excel nao desmembrou.

A estrutura e **4 canais de 4 bytes**, cada canal um inteiro de 16 bits:

```
[2,191,100,0]  [0,55,1,0]  [0,192,1,0]  [0,32,1,0]
 canal 1        canal 2     canal 3      canal 4
 V = 7.03       A = 0.55    X = 1.92     T = 32
   ^              ^           ^            ^
 coluna 4      coluna 5     PERDIDO     coluna 6
```

> **Pergunta rapida:** por que mexer nos bytes se ja existem colunas prontas?
> **Resposta:** porque o **canal 3 nunca foi exportado** - e ele e o unico que
> nao e copia linear da velocidade.

In [ ]:
def decodificar_payload(coluna_bytes):
    """Converte a string de bytes do IO-Link em 4 canais numericos."""
    B = np.array([[int(x) for x in s.split(",")] for s in coluna_bytes])
    return pd.DataFrame({
        "V": (B[:, 0] * 256 + B[:, 1]) / 100,   # canal 1 - velocidade de vibracao
        "A":  B[:, 5] / 100,                    # canal 2 - aceleracao
        "X": (B[:, 8] * 256 + B[:, 9]) / 100,   # canal 3 - NAO exportado no Excel
        "T":  B[:, 13].astype(float),           # canal 4 - temperatura
    })

canais = {}
for port, motor in [("pdi_motor1", 1), ("pdi_motor2", 2)]:
    d = decodificar_payload(df_raw[port])
    d.index = df_raw["ts"]
    canais[motor] = d

print("Canal 3 (X) - recuperado do payload:")
display(pd.DataFrame({
    "motor 1": canais[1]["X"].describe(),
    "motor 2": canais[2]["X"].describe(),
}).round(2))

In [ ]:
# Quanta informacao independente cada canal carrega, em relacao a velocidade?
d1 = canais[1]
em_operacao = d1["V"] > 1

linhas = []
for c in ["A", "X", "T"]:
    r2 = np.corrcoef(d1.loc[em_operacao, "V"], d1.loc[em_operacao, c])[0, 1] ** 2
    linhas.append({"canal": c, "R2 vs Velocidade": round(r2, 3),
                   "variancia independente": f"{(1-r2)*100:.1f}%"})

print("Redundancia dos canais (apenas amostras em operacao):")
display(pd.DataFrame(linhas))

### Leitura

- **Aceleracao**: R2 = 0,97 contra velocidade. E velocidade multiplicada por uma
  constante - **descartavel** para o clustering.
- **Canal X (recuperado)**: R2 = 0,65. Sobram ~35% de variancia independente.
  E o unico canal mecanico com informacao nova.
- **Temperatura**: independente, como esperado.

**Decisao:** o K-Means vai usar `V`, `X` e `T`. A aceleracao fica de fora por
redundancia - e isso e registrado como decisao de EDA, nao como omissao.

## 5) Grade temporal regular

**O que esta etapa faz:** a amostragem original e irregular (de 0,35 s a 228 s entre
leituras). Toda estatistica de janela movel assume passo constante - sem reamostrar,
"media dos ultimos 30 s" significa coisas diferentes em cada linha.

> **Pergunta rapida:** por que 5 s?
> **Resposta:** e proximo da mediana real de amostragem e mantem a serie leve sem
> perder a dinamica do processo.

In [ ]:
delta = df_raw.ts.diff().dt.total_seconds()
print(f"Intervalo entre amostras - mediana: {delta.median():.2f}s | "
      f"media: {delta.mean():.2f}s | maximo: {delta.max():.1f}s")

PASSO = "5s"
series = {}
for motor, d in canais.items():
    series[motor] = d.sort_index().resample(PASSO).mean().interpolate()
    print(f"Motor {motor}: {len(series[motor])} pontos em grade regular de {PASSO}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for motor, d in series.items():
    axes[0].plot(d.index, d["V"], lw=.9, label=f"Motor {motor}")
    axes[1].plot(d.index, d["X"], lw=.9, label=f"Motor {motor}")
    axes[2].plot(d.index, d["T"], lw=.9, label=f"Motor {motor}")

axes[0].set_ylabel("V - vibracao")
axes[1].set_ylabel("X - canal recuperado")
axes[2].set_ylabel("T - temperatura (C)")
axes[2].set_xlabel("Horario")
axes[0].set_title("Visualizacao inicial: os tres canais independentes")
for ax in axes:
    ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

### Leitura

Seis partidas e seis paradas na sessao inteira. As corridas duram
1,6 / 1,3 / 1,4 / 3,8 / 3,8 e **16,9 min**. A temperatura sobe durante a operacao
e cai no repouso.

Guarde esse numero: **6 ciclos**. Ele volta na conclusao.

## 6) BASE 1 - a janela futura

**O que esta etapa faz:** constroi o equivalente a *Base 1* da Ford - o comportamento
**observado** do ativo. Na Ford era o historico de 24 meses pos-venda. Aqui e o que
acontece nos **proximos 10 minutos**.

| variavel | significado |
|---|---|
| `fut_T_max`   | maior temperatura atingida no horizonte |
| `fut_T_delta` | variacao de temperatura no horizonte |
| `fut_V_max`   | pico de vibracao |
| `fut_V_mean`  | vibracao media |
| `fut_A_max`   | pico de aceleracao |
| `fut_X_max`   | pico do canal recuperado |

> **Pergunta rapida:** por que deixar a analise escolher, em vez de eu definir que
> "temperatura e o alvo"?
> **Resposta:** porque um motor pode degradar por calor **ou** por vibracao.
> Clusterizando a janela futura inteira, e o **K-Means** que revela qual variavel
> separa os perfis - nao uma suposicao minha.

In [ ]:
HORIZONTE_PASSOS = 120           # 120 x 5s = 10 minutos
HORIZONTE_MIN = HORIZONTE_PASSOS * 5 / 60

def construir_base1(d):
    """Estatisticas da janela FUTURA (proximos HORIZONTE_PASSOS pontos)."""
    rev = d.iloc[::-1]                        # rolling reverso = janela futura
    def fut(col, fn):
        r = rev[col].rolling(HORIZONTE_PASSOS, min_periods=HORIZONTE_PASSOS)
        return getattr(r, fn)().iloc[::-1]

    f = pd.DataFrame(index=d.index)
    f["fut_T_max"]   = fut("T", "max")
    f["fut_T_delta"] = d["T"].shift(-HORIZONTE_PASSOS) - d["T"]
    f["fut_V_max"]   = fut("V", "max")
    f["fut_V_mean"]  = fut("V", "mean")
    f["fut_A_max"]   = fut("A", "max")
    f["fut_X_max"]   = fut("X", "max")
    return f

print(f"Horizonte de previsao: {HORIZONTE_MIN:.0f} minutos")

## 7) BASE 2 - a janela passada

**O que esta etapa faz:** constroi o equivalente a *Base 2* da Ford - so o que o
sistema **sabe agora**. Na Ford, informacao do momento da compra. Aqui, os ultimos
30 s / 2 min / 10 min.

**Regra de validade (slide 18):** proibido usar qualquer estatistica da janela futura
como entrada. A verificacao dessa disjuncao e feita explicitamente na etapa 10.

In [ ]:
JANELAS = [(6, "30s"), (24, "2min"), (120, "10min")]

def construir_base2(d):
    """Features do PASSADO apenas - disponiveis no instante da decisao."""
    f = pd.DataFrame(index=d.index)

    for c in ["V", "A", "X", "T"]:
        f[f"{c}_agora"] = d[c]
        for w, nome in JANELAS:
            r = d[c].rolling(w, min_periods=1)
            f[f"{c}_media_{nome}"] = r.mean()
            f[f"{c}_std_{nome}"]   = r.std().fillna(0)
            f[f"{c}_max_{nome}"]   = r.max()

    # tendencia termica recente
    f["T_slope_2min"]  = d["T"].diff(24).fillna(0)
    f["T_slope_10min"] = d["T"].diff(120).fillna(0)

    # contexto operacional
    rodando = (d["V"] > 1).astype(int)
    grupo = (rodando.diff() != 0).cumsum()
    f["rodando"] = rodando
    f["tempo_no_estado_min"] = rodando.groupby(grupo).cumcount() * 5 / 60
    f["carga_termica"] = (d["V"] * 5).groupby(grupo).cumsum()
    return f

# Monta o dataset final: um registro por (motor, instante)
blocos = []
for motor, d in series.items():
    bloco = pd.concat([construir_base1(d), construir_base2(d)], axis=1)
    bloco["motor"] = motor
    blocos.append(bloco.reset_index().rename(columns={"index": "ts"}))

dados = pd.concat(blocos, ignore_index=True).dropna().sort_values("ts").reset_index(drop=True)

FEATURES_B1 = [c for c in dados.columns if c.startswith("fut_")]
FEATURES_B2 = [c for c in dados.columns if not c.startswith("fut_") and c != "ts"]

print("Dataset final:", dados.shape)
print(f"Base 1 (clusterizacao): {len(FEATURES_B1)} variaveis")
print(f"Base 2 (classificacao): {len(FEATURES_B2)} variaveis")
display(dados[["ts", "motor"] + FEATURES_B1].head())

## 8) Padronizar e escolher `k`

> **Pergunta rapida:** por que padronizar?
> **Resposta:** porque o K-Means trabalha com **distancias**. Temperatura vai a 46 e
> aceleracao a 0,58 - sem padronizar, a temperatura domina o agrupamento sozinha.

Usamos **dois** criterios: o **cotovelo** (inercia/SSE) e a **silhueta**.
O cotovelo orienta; ele nao decide sozinho.

In [ ]:
scaler_b1 = StandardScaler()
X_b1 = scaler_b1.fit_transform(dados[FEATURES_B1])

valores_k = range(2, 8)
inercias, silhuetas = [], []
for k in valores_k:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_b1)
    inercias.append(km.inertia_)
    silhuetas.append(silhouette_score(X_b1, km.labels_, sample_size=3000,
                                      random_state=RANDOM_STATE))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(list(valores_k), inercias, marker="o")
ax1.set_xlabel("Numero de clusters (k)"); ax1.set_ylabel("Inercia / SSE")
ax1.set_title("Metodo do cotovelo")
ax2.plot(list(valores_k), silhuetas, marker="o", color="darkorange")
ax2.set_xlabel("Numero de clusters (k)"); ax2.set_ylabel("Silhueta media")
ax2.set_title("Coeficiente de silhueta")
plt.tight_layout(); plt.show()

display(pd.DataFrame({"k": list(valores_k),
                      "inercia": np.round(inercias, 0),
                      "silhueta": np.round(silhuetas, 3)}))

### Justificativa de `k`

- O cotovelo mostra ganho decrescente a partir de `k = 4`.
- A silhueta e maxima em `k = 3`, mas `k = 4` continua acima de 0,45 - separacao solida.
- `k = 4` e **interpretavel em linguagem de operacao**: repouso frio, repouso quente,
  operacao leve e operacao intensa.

Como na aula: *k e uma hipotese que precisa ser tecnicamente defensavel **e** util
para o negocio*. Escolhemos **`k = 4`**.

## 9) Treinar o K-Means e interpretar os perfis

> **Pergunta rapida:** o K-Means ja da nomes como "critico" ou "saudavel"?
> **Resposta:** nao. Ele entrega **grupos numericos**. O nome vem depois, comparando
> as medias e justificando a leitura de operacao.

In [ ]:
K = 4
kmeans = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10)
dados["cluster"] = kmeans.fit_predict(X_b1)

perfil = dados.groupby("cluster")[FEATURES_B1].mean().round(2)
tamanho = dados["cluster"].value_counts().sort_index().rename("n_amostras")

print("Perfil medio de cada cluster (janela futura):")
display(perfil.join(tamanho))

print("\nCentros na escala original:")
display(pd.DataFrame(scaler_b1.inverse_transform(kmeans.cluster_centers_),
                     columns=FEATURES_B1).round(2))

In [ ]:
# Qual variavel separa cada cluster dos demais?
# Centroide em unidades padronizadas = distancia a media geral em desvios-padrao.
assinatura = pd.DataFrame(kmeans.cluster_centers_, columns=FEATURES_B1).round(2)

plt.figure(figsize=(10, 4.5))
sns.heatmap(assinatura, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            cbar_kws={"label": "desvios-padrao da media geral"})
plt.title("Assinatura relativa dos perfis - o que separa cada cluster")
plt.ylabel("cluster"); plt.tight_layout(); plt.show()

### Traducao semantica dos perfis

Como no slide 13 da aula - *"Cluster 0 nao e um resultado executivo"*.
Interprete a tabela acima e nomeie os perfis. A leitura tipica desta sessao:

Com `RANDOM_STATE = 42`, a saida observada foi:

| cluster | T_max | dT | V_max | V_med | assinatura | nome operacional |
|---|---|---|---|---|---|---|
| **2** | 33,6 | +0,3  | 0,19 | 0,05 | parado e frio | **Repouso estavel** |
| **0** | 43,2 | -0,9  | 0,09 | 0,04 | parado porem quente, esfriando | **Resfriamento pos-carga** |
| **1** | 32,7 | -0,5  | 6,48 | 1,62 | picos de vibracao sem aquecer | **Operacao intermitente** |
| **3** | 40,9 | **+5,9** | 6,77 | 4,78 | vibracao sustentada + forte aquecimento | **Operacao critica** |

O que separa os perfis nao e uma variavel so: **`fut_V_mean` separa operacao de
repouso** e **`fut_T_delta` separa quem aquece de quem esfria**. Note que os clusters
1 e 3 tem `V_max` quase identico (6,48 vs 6,77) - o que os distingue e a vibracao
*sustentada* (`V_mean` 1,62 vs 4,78) e o aquecimento resultante. Pico de vibracao
sozinho nao caracteriza risco; vibracao mantida sim.

> Se mudar a semente, a numeracao dos clusters muda. Guie-se pela assinatura, nao pelo indice.

O perfil **Operacao critica** e o alvo de negocio: e nele que o ativo acumula
carga termica. Antecipar a entrada nesse perfil e o objetivo da Etapa 2.

In [ ]:
# Visualizacao dos perfis reduzida por PCA (como no slide 11)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
proj = pca.fit_transform(X_b1)

plt.figure(figsize=(9, 6))
sns.scatterplot(x=proj[:, 0], y=proj[:, 1], hue=dados["cluster"],
                palette="tab10", alpha=.45, s=22)
centros_proj = pca.transform(kmeans.cluster_centers_)
plt.scatter(centros_proj[:, 0], centros_proj[:, 1], c="black", s=220,
            marker="X", label="Centroides", zorder=5)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% da variancia)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% da variancia)")
plt.title("Perfis de comportamento futuro segmentados pelo K-Means")
plt.legend(title="cluster"); plt.tight_layout(); plt.show()

# ETAPA 2 - Da segmentacao a classificacao

> *"A saida do clustering vira target supervisionado."* (slide 17)

Agora a pergunta muda de natureza:

- **Antes:** nao havia `y`. O K-Means criou rotulos operacionais.
- **Agora:** a Base 2 ganha um `y` e o problema vira **classificacao multiclasse**.

`cluster_id -> perfil operacional`
`0/1/2/3 -> Repouso / Critica / Resfriamento / Intermitente`

## 10) Criar o target e verificar o vazamento

**O que esta etapa faz:** o passo mais importante do notebook. Confirma que as
features do classificador **nao contem** nenhuma variavel usada na clusterizacao.

In [ ]:
dados["y"] = dados["cluster"]

vazamento = set(FEATURES_B1) & set(FEATURES_B2)
print("Variaveis em comum entre Base 1 e Base 2:", vazamento if vazamento else "NENHUMA")
assert not vazamento, "VAZAMENTO: a Base 2 contem variaveis da janela futura!"

futuras_na_b2 = [c for c in FEATURES_B2 if c.startswith("fut_")]
print("Features futuras dentro da Base 2:", futuras_na_b2 if futuras_na_b2 else "NENHUMA")
assert not futuras_na_b2

print("\nOK - os conjuntos sao disjuntos.")
print(f"  Base 1 (gera o target):   {len(FEATURES_B1)} variaveis, todas do FUTURO")
print(f"  Base 2 (entra no modelo): {len(FEATURES_B2)} variaveis, todas do PASSADO/PRESENTE")
print("\nDistribuicao do target:")
display(dados["y"].value_counts().sort_index().rename("n"))

## 11) Treinar - split aleatorio (o procedimento da aula)

**O que esta etapa faz:** reproduz exatamente o `train_test_split` do notebook da aula:
70% treino, 30% teste, estratificado.

In [ ]:
treino_a, teste_a = train_test_split(
    dados, test_size=0.30, random_state=RANDOM_STATE, stratify=dados["y"])

sc_a = StandardScaler().fit(treino_a[FEATURES_B2])
Xtr_a, Xte_a = sc_a.transform(treino_a[FEATURES_B2]), sc_a.transform(teste_a[FEATURES_B2])
ytr_a, yte_a = treino_a["y"], teste_a["y"]

modelos = {
    "Regressao Logistica": LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
    "Arvore de Decisao":   DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
}

res_aleatorio = []
for nome, m in modelos.items():
    m.fit(Xtr_a, ytr_a)
    p = m.predict(Xte_a)
    res_aleatorio.append({
        "modelo": nome,
        "accuracy":  accuracy_score(yte_a, p),
        "precision": precision_score(yte_a, p, average="macro", zero_division=0),
        "recall":    recall_score(yte_a, p, average="macro", zero_division=0),
        "f1_macro":  f1_score(yte_a, p, average="macro"),
    })

baseline_a = [ytr_a.mode()[0]] * len(yte_a)
res_aleatorio.append({
    "modelo": "BASELINE (classe majoritaria)",
    "accuracy":  accuracy_score(yte_a, baseline_a),
    "precision": precision_score(yte_a, baseline_a, average="macro", zero_division=0),
    "recall":    recall_score(yte_a, baseline_a, average="macro", zero_division=0),
    "f1_macro":  f1_score(yte_a, baseline_a, average="macro"),
})

tab_aleatorio = pd.DataFrame(res_aleatorio).set_index("modelo").round(3)
print("=== SPLIT ALEATORIO ===")
display(tab_aleatorio)

In [ ]:
melhor = DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE).fit(Xtr_a, ytr_a)
pred_a = melhor.predict(Xte_a)

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(yte_a, pred_a), annot=True, fmt="d", cmap="Blues",
            cbar=False)
plt.xlabel("Previsto"); plt.ylabel("Real")
plt.title("Matriz de confusao - split aleatorio")
plt.tight_layout(); plt.show()

print(classification_report(yte_a, pred_a, zero_division=0))

### Leitura

Numeros altos. Pela regua da aula, o trabalho estaria concluido aqui.

**Mas ha um problema, e ele e o tema do slide 2: vazamento de dados.**

A telemetria e amostrada a cada 5 s. Duas linhas consecutivas sao quase identicas.
O `train_test_split` embaralha e separa **linhas vizinhas** entre treino e teste -
o modelo e avaliado em amostras que praticamente ja viu.

Em producao o modelo nunca ve o futuro. A avaliacao honesta precisa **cortar no tempo**.

## 12) Treinar - split temporal (a avaliacao honesta)

**O que esta etapa faz:** usa `TimeSeriesSplit`. Cada fold treina so no passado e
testa so no futuro. Nenhuma linha do teste e vizinha de uma linha do treino.

In [ ]:
res_temporal = []
tss = TimeSeriesSplit(n_splits=5)

for i, (idx_tr, idx_te) in enumerate(tss.split(dados)):
    tr, te = dados.iloc[idx_tr], dados.iloc[idx_te]
    sc = StandardScaler().fit(tr[FEATURES_B2])
    Xtr, Xte = sc.transform(tr[FEATURES_B2]), sc.transform(te[FEATURES_B2])

    linha = {"fold": i, "n_teste": len(te), "classes_no_teste": te["y"].nunique()}
    for nome, m in modelos.items():
        m.fit(Xtr, tr["y"])
        linha[nome] = f1_score(te["y"], m.predict(Xte), average="macro")
    linha["BASELINE"] = f1_score(te["y"], [tr["y"].mode()[0]] * len(te),
                                 average="macro")
    res_temporal.append(linha)

tab_temporal = pd.DataFrame(res_temporal)
print("=== SPLIT TEMPORAL (f1_macro) ===")
display(tab_temporal.round(3))
print("\nMedia entre folds:")
display(tab_temporal[list(modelos) + ["BASELINE"]].mean().round(3).rename("f1_macro medio"))

In [ ]:
comparacao = pd.DataFrame({
    "split aleatorio (aula)": [tab_aleatorio.loc["Regressao Logistica", "f1_macro"],
                               tab_aleatorio.loc["Arvore de Decisao", "f1_macro"]],
    "split temporal (honesto)": [tab_temporal["Regressao Logistica"].mean(),
                                 tab_temporal["Arvore de Decisao"].mean()],
}, index=["Regressao Logistica", "Arvore de Decisao"]).round(3)
comparacao["queda"] = (comparacao.iloc[:, 0] - comparacao.iloc[:, 1]).round(3)

print("=== O CUSTO DO VAZAMENTO ===")
display(comparacao)

comparacao[["split aleatorio (aula)", "split temporal (honesto)"]].plot(
    kind="bar", figsize=(8, 4.5), rot=0, color=["#7aa6c2", "#c2705a"])
plt.ylabel("f1_macro"); plt.ylim(0, 1)
plt.title("Mesmo modelo, mesmas features - so muda como o teste e montado")
plt.tight_layout(); plt.show()

## 13) Por que a queda e tao grande?

**O que esta etapa faz:** localiza a causa raiz. Nao e hiperparametro - e cobertura
de dados.

In [ ]:
dados["bloco"] = pd.cut(dados["ts"].astype("int64"), 6, labels=False)
cobertura = pd.crosstab(dados["bloco"], dados["y"])

plt.figure(figsize=(8, 4))
sns.heatmap(cobertura, annot=True, fmt="d", cmap="YlOrRd", cbar_kws={"label": "amostras"})
plt.xlabel("perfil (cluster)"); plt.ylabel("bloco temporal da sessao")
plt.title("Quais perfis existem em cada trecho da sessao")
plt.tight_layout(); plt.show()

display(cobertura)
print("\nPerfis distintos presentes em cada bloco:")
display((cobertura > 20).sum(axis=1).rename("n_perfis"))

### Diagnostico

Os perfis **aparecem e desaparecem** ao longo da sessao. Alguns so existem na
segunda metade. Um modelo treinado no passado e avaliado sobre perfis que
**nunca apareceram no treino** - e nenhum algoritmo resolve isso.

A causa esta na base: **6 ciclos de operacao, uma unica sessao de 4 h, um unico dia**.
Do ponto de vista estatistico, o numero de exemplos independentes nao e 5.450 -
e da ordem de **6**.

Nao e um defeito do pipeline. E o limite do dado disponivel.

## 14) Exportar os resultados

In [ ]:
saida = dados[["ts", "motor", "y"] + FEATURES_B1].copy()
saida = saida.rename(columns={"y": "perfil"})
saida.to_csv("forzy_perfis_segmentados.csv", index=False)
print("Exportado: forzy_perfis_segmentados.csv", saida.shape)
display(saida.head())

## 15) Conclusao

### O que foi entregue

1. **Recuperacao de um canal de sensor perdido na exportacao** (canal 3 do payload
   IO-Link) - o unico com variancia independente da velocidade.
2. **Segmentacao nao supervisionada** da janela futura em 4 perfis operacionais
   interpretaveis, com `k` justificado por cotovelo **e** silhueta.
3. **Conversao dos clusters em target** e treino de classificadores supervisionados,
   com separacao estrita entre variaveis do futuro (Base 1) e do passado (Base 2).
4. **Dupla avaliacao** - o split da aula e o split temporal - quantificando o
   vazamento de dados.

### O que o modelo pode e nao pode afirmar

| Afirmacao | Sustentada? |
|---|---|
| "Os canais do sensor sao majoritariamente redundantes; existe um 4o canal nao exportado" | **Sim** - verificavel nos bytes |
| "O comportamento futuro se organiza em 4 perfis operacionais distintos" | **Sim** - silhueta > 0,45 |
| "O pipeline segmentacao -> target -> classificacao esta livre de circularidade" | **Sim** - conjuntos disjuntos, verificado por assercao |
| "O classificador preve o perfil com ~0,94 de F1" | **Nao** - vem do split aleatorio, inflado por vazamento |
| "O classificador esta pronto para producao" | **Nao** - F1 temporal ~0,33 |
| "O modelo preve falha do motor" | **Nao** - nao ha nenhuma falha rotulada na base |

### Limitacoes

- Uma sessao, um dia, ~4 h, **6 ciclos** de operacao.
- Os dois motores sao acionados juntos (correlacao ~1,0) - nao sao amostras independentes.
- **Nenhuma falha registrada.** O pipeline segmenta *regimes de operacao*, nao
  *estados de saude*. Qualquer rotulo do tipo "motor com defeito" seria invencao.
- A unidade fisica dos canais depende do IODD do sensor. Se `V` for velocidade de
  vibracao em mm/s, a **ISO 10816** fornece faixas normativas de severidade - o que
  daria rotulos de referencia externa em vez de rotulos derivados do proprio dado.

### Proximo passo

Coletar **multiplas sessoes, em dias diferentes, com mais ciclos de partida e parada**.
O pipeline esta pronto e validado; falta base. Com 30-50 ciclos distribuidos em varios
dias, o split temporal passa a ter cobertura de classes e o F1 honesto vira um numero
utilizavel.